# Can we predict next-month account churn?

## Executive answer

**Yes, with useful but moderate ranking power—provided we remove a severe data leak.**

On the untouched September–November 2025 holdout, the selected model achieves:

- **ROC AUC 0.750** (account-clustered 95% interval: **0.716–0.785**)
- **Average precision 0.285**, versus a **0.097** no-skill baseline
- The top-scored **10%** of account-months contain **35.5%** of churners, with **34.4% precision** (3.55× lift)
- Mean predicted risk is **10.1%**, close to the observed **9.7%**

This supports a targeted retention workflow, not fully automated decisions. The data has substantial time drift, and the result comes from one historical holdout period; prospective monitoring is required.

The column `lifetime_months` is **not usable**: it was calculated from a 2026 export and perfectly reveals the outcome. Including it would produce a fake near-perfect result.

## 1. Setup and data contract

The prediction moment is the end of `snapshot_month`; the outcome is cancellation in the following month. Therefore only information available by month-end may be used. We use a forward-in-time evaluation rather than a random row split.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 100)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_PATH = Path("data.csv")
DICTIONARY_PATH = Path("columns.csv")
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
dictionary = pd.read_csv(DICTIONARY_PATH)
df["snapshot_date"] = pd.to_datetime(df["snapshot_month"], format="%Y-%m")

print(f"Rows: {len(df):,} | Accounts: {df['account_ref'].nunique():,} | Months: {df['snapshot_month'].nunique()}")
display(dictionary)

## 2. Data quality and target behavior

We check schema validity, missingness, duplicate account-months, class balance, and prevalence over time. The temporal prevalence plot is particularly important because changing base rates can damage probability calibration even when ranking remains useful.

In [ ]:
expected_columns = set(dictionary["column"])
assert expected_columns.issubset(df.columns), "Dataset does not match the supplied dictionary."
assert set(df["churn_next_month"].unique()) <= {0, 1}, "Target must be binary."

quality = pd.Series({
    "rows": len(df),
    "accounts": df["account_ref"].nunique(),
    "months": df["snapshot_month"].nunique(),
    "missing cells": int(df[list(expected_columns)].isna().sum().sum()),
    "duplicate account-months": int(df.duplicated(["account_ref", "snapshot_month"]).sum()),
    "overall churn rate": df["churn_next_month"].mean(),
}, name="value")
display(quality.to_frame())

monthly = (
    df.groupby("snapshot_date")["churn_next_month"]
      .agg(rows="size", churners="sum", churn_rate="mean")
      .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly["snapshot_date"], monthly["churn_rate"], marker="o", color="#3157a4")
ax.axvspan(pd.Timestamp("2025-09-01"), pd.Timestamp("2025-11-30"), color="#f2b134", alpha=0.2, label="final holdout")
ax.set(title="Churn prevalence changes sharply over time", xlabel="Snapshot month", ylabel="Churn next month")
ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.legend()
plt.show()

display(monthly.assign(churn_rate=monthly["churn_rate"].map(lambda x: f"{x:.1%}")))

## 3. Leakage audit

`lifetime_months` is described as total observed lifetime from a **2026-01-05 CRM export**. That information occurs after nearly every prediction date. It is therefore future information, not a legitimate feature.

Below we test the relationship explicitly. `account_ref` is also excluded because it is an identifier rather than a generalizable account attribute.

In [ ]:
leak_rule = (df["lifetime_months"] == df["tenure_months"]).astype(int)
leakage_check = pd.Series({
    "Rows where leak rule matches target": (leak_rule == df["churn_next_month"]).mean(),
    "ROC AUC of leak rule": roc_auc_score(df["churn_next_month"], leak_rule),
}, name="value")
display(leakage_check.to_frame())

assert (leak_rule == df["churn_next_month"]).all()
print("Conclusion: lifetime_months perfectly reconstructs the target and must be excluded.")

## 4. Evaluation design

We preserve chronology:

- **Model-development training:** data through June 2025
- **Validation:** July–August 2025, used to choose a model/training window
- **Final untouched test:** September–November 2025

The prevalence spike in late 2024/early 2025 represents a different regime. We compare all-history training with a recent-regime window beginning March 2025. The final test is only evaluated after the choice.

Metrics:

- **ROC AUC:** ranking quality, insensitive to a chosen threshold
- **Average precision (AP):** ranking focused on the minority churn class; no-skill AP equals prevalence
- **Brier score / log loss:** probability quality; lower is better

The same account can appear across periods, which matches monthly production scoring. The identifier is not modeled, and final confidence intervals resample whole accounts to respect within-account dependence.

In [ ]:
target = "churn_next_month"
features = ["plan", "seats", "monthly_fee", "tenure_months", "usage_hours", "support_tickets"]
categorical = ["plan"]
numeric = [c for c in features if c not in categorical]

preprocessor = ColumnTransformer([
    ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ("number", StandardScaler(), numeric),
])

def logistic_model():
    return make_pipeline(preprocessor, LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE))

def boosted_model():
    return make_pipeline(
        preprocessor,
        HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
            l2_regularization=1.0, random_state=RANDOM_STATE,
        ),
    )

def score_probabilities(y_true, probability):
    return {
        "ROC AUC": roc_auc_score(y_true, probability),
        "Average precision": average_precision_score(y_true, probability),
        "Brier score": brier_score_loss(y_true, probability),
        "Log loss": log_loss(y_true, probability),
        "Mean prediction": np.mean(probability),
        "Observed prevalence": np.mean(y_true),
    }

development_train = df["snapshot_month"] <= "2025-06"
recent_train = df["snapshot_month"].between("2025-03", "2025-06")
validation = df["snapshot_month"].between("2025-07", "2025-08")
final_test = df["snapshot_month"].between("2025-09", "2025-11")

split_summary = pd.DataFrame({
    "rows": [development_train.sum(), recent_train.sum(), validation.sum(), final_test.sum()],
    "churn rate": [
        df.loc[development_train, target].mean(),
        df.loc[recent_train, target].mean(),
        df.loc[validation, target].mean(),
        df.loc[final_test, target].mean(),
    ],
}, index=["all-history dev train", "recent-regime train", "validation", "final test"])
display(split_summary.style.format({"churn rate": "{:.1%}"}))

## 5. Select the approach on validation only

The dummy baseline estimates a constant recent prevalence. The other candidates use identical safe features, varying model complexity and whether they learn from the older, shifted regime.

In [ ]:
candidates = [
    ("Dummy (recent prevalence)", make_pipeline(preprocessor, DummyClassifier(strategy="prior")), recent_train),
    ("Logistic (all history)", logistic_model(), development_train),
    ("Gradient boosting (all history)", boosted_model(), development_train),
    ("Logistic (recent regime)", logistic_model(), recent_train),
    ("Gradient boosting (recent regime)", boosted_model(), recent_train),
]

validation_rows = []
for name, model, train_mask in candidates:
    model.fit(df.loc[train_mask, features], df.loc[train_mask, target])
    probability = model.predict_proba(df.loc[validation, features])[:, 1]
    validation_rows.append({"Model": name, **score_probabilities(df.loc[validation, target], probability)})

validation_results = pd.DataFrame(validation_rows).set_index("Model").sort_values("Brier score")
display(validation_results.style.format("{:.3f}"))

selected_name = "Logistic (recent regime)"
print(f"Selected before touching the final test: {selected_name}")

The recent-regime logistic model gives the best validation discrimination among the practical candidates and materially better-calibrated probabilities than all-history training. Its simplicity is also appropriate for this small tabular dataset.

## 6. One-time final holdout evaluation

We now retrain the selected specification on March–August 2025, then evaluate once on September–November. No threshold is tuned on this holdout.

In [ ]:
final_train = df["snapshot_month"].between("2025-03", "2025-08")
final_model = logistic_model()
final_model.fit(df.loc[final_train, features], df.loc[final_train, target])

X_test = df.loc[final_test, features]
y_test = df.loc[final_test, target].to_numpy()
p_test = final_model.predict_proba(X_test)[:, 1]
final_metrics = score_probabilities(y_test, p_test)

baseline_ap = y_test.mean()
metrics_table = pd.DataFrame({
    "Selected model": final_metrics,
    "Reference": {
        "ROC AUC": 0.5,
        "Average precision": baseline_ap,
        "Brier score": y_test.mean() * (1 - y_test.mean()),
        "Log loss": np.nan,
        "Mean prediction": y_test.mean(),
        "Observed prevalence": y_test.mean(),
    },
})
display(metrics_table.style.format("{:.3f}"))

print(f"Test rows: {final_test.sum():,}; churners: {y_test.sum():,}; accounts: {df.loc[final_test, 'account_ref'].nunique():,}")

### Account-clustered uncertainty

Because accounts contribute repeated monthly rows, ordinary row-level intervals would be too optimistic. This bootstrap samples accounts—not individual rows—with replacement.

In [ ]:
test_predictions = df.loc[final_test, ["account_ref", "snapshot_month", target]].copy()
test_predictions["predicted_churn"] = p_test

groups = [
    group[[target, "predicted_churn"]].to_numpy()
    for _, group in test_predictions.groupby("account_ref", sort=False)
]
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_scores = []

for _ in range(1_000):
    sample = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
    y_boot, p_boot = sample[:, 0], sample[:, 1]
    bootstrap_scores.append([
        roc_auc_score(y_boot, p_boot),
        average_precision_score(y_boot, p_boot),
        brier_score_loss(y_boot, p_boot),
    ])

interval = pd.DataFrame(
    np.quantile(bootstrap_scores, [0.025, 0.5, 0.975], axis=0),
    index=["2.5%", "median", "97.5%"],
    columns=["ROC AUC", "Average precision", "Brier score"],
)
display(interval.style.format("{:.3f}"))

### Operational lift

Ranking metrics become concrete when translated into outreach capacity. The table below asks what happens if a retention team contacts only the top-scored fraction of account-months.

In [ ]:
order = np.argsort(-p_test)
capacity_rows = []
for fraction in [0.05, 0.10, 0.20, 0.30]:
    n_contact = int(len(y_test) * fraction)
    selected = order[:n_contact]
    precision = y_test[selected].mean()
    recall = y_test[selected].sum() / y_test.sum()
    capacity_rows.append({
        "Contact fraction": fraction,
        "Accounts-months contacted": n_contact,
        "Precision": precision,
        "Share of churners captured": recall,
        "Lift vs random": precision / y_test.mean(),
    })

capacity = pd.DataFrame(capacity_rows)
display(capacity.style.format({
    "Contact fraction": "{:.0%}",
    "Precision": "{:.1%}",
    "Share of churners captured": "{:.1%}",
    "Lift vs random": "{:.2f}×",
}))

precision_curve, recall_curve, _ = precision_recall_curve(y_test, p_test)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(recall_curve, precision_curve, color="#3157a4", lw=2, label=f"Model (AP={final_metrics['Average precision']:.3f})")
axes[0].axhline(y_test.mean(), color="#ba3c3c", ls="--", label=f"No-skill ({y_test.mean():.3f})")
axes[0].set(title="Precision–recall on final holdout", xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1))
axes[0].legend()

calibration = pd.DataFrame({"y": y_test, "p": p_test})
calibration["bin"] = pd.qcut(calibration["p"], q=10, duplicates="drop")
calibration = calibration.groupby("bin", observed=True).agg(mean_prediction=("p", "mean"), observed_rate=("y", "mean"), rows=("y", "size"))
axes[1].plot([0, 1], [0, 1], color="gray", ls="--", label="Perfect calibration")
axes[1].plot(calibration["mean_prediction"], calibration["observed_rate"], marker="o", color="#3157a4", label="Model")
axes[1].set(title="Calibration by score decile", xlabel="Mean predicted probability", ylabel="Observed churn rate", xlim=(0, 0.5), ylim=(0, 0.5))
axes[1].legend()
plt.tight_layout()
plt.show()

display(calibration.style.format({"mean_prediction": "{:.1%}", "observed_rate": "{:.1%}"}))

## 7. Interpretation

For numeric variables, odds ratios below correspond to a one-standard-deviation increase. Plan effects are relative contributions under one-hot encoding and should be read directionally, not causally. These are associations, not proof that changing a feature will prevent churn.

In [ ]:
feature_names = final_model.named_steps["columntransformer"].get_feature_names_out()
coefficients = final_model.named_steps["logisticregression"].coef_[0]

coefficient_table = pd.DataFrame({
    "feature": [name.replace("category__", "").replace("number__", "") for name in feature_names],
    "coefficient": coefficients,
    "odds ratio": np.exp(coefficients),
}).sort_values("coefficient")

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = np.where(coefficient_table["coefficient"] > 0, "#ba3c3c", "#3157a4")
ax.barh(coefficient_table["feature"], coefficient_table["coefficient"], color=colors)
ax.axvline(0, color="black", lw=1)
ax.set(title="Logistic model associations", xlabel="Coefficient (positive means higher predicted churn)")
plt.tight_layout()
plt.show()

display(coefficient_table.sort_values("coefficient", ascending=False).style.format({"coefficient": "{:.3f}", "odds ratio": "{:.2f}×"}))

## 8. Decision and next steps

**Decision: proceed to a monitored retention pilot.** The model separates risk meaningfully: its average precision is about three times the holdout prevalence, and a 10% outreach budget captures about 35% of observed churners. Calibration is also reasonable in aggregate for the held-out period.

Before production:

1. Confirm the scoring timestamp and that all six features are finalized by month-end.
2. Permanently block `lifetime_months` and other post-outcome CRM fields from the feature pipeline.
3. Run a prospective month or an A/B retention experiment; prediction quality alone does not show that outreach changes outcomes.
4. Monitor monthly prevalence, ROC AUC/AP, calibration, and top-decile lift. Retrain on a recent window when drift is material.
5. Choose an outreach threshold from intervention capacity and economics—not from a default 0.5 cutoff.

Limitations: there are only 23 months, churn prevalence undergoes a sharp regime shift, and the final estimate covers three months. The result is evidence of useful signal, not a guarantee of future stability.